<a href="https://colab.research.google.com/github/falawar7/AAI614_ELAWAR/blob/main/Notebook/Import_export_normalize_CSV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import re
import numpy as np

df_new_25 = pd.read_csv("/content/drive/Othercomputers/THINKPAD-T14/AI/Project/Rev2/Colab/24-25.csv")
#df_new_25['Course_Name'] = df_new_25['Course_Name'].str.replace(r'\s+\d+$', '', regex=True)

#normalize the Terms and Electives  of Semester 2021-2022 for df1 and 2022-2023 for df2
def normalize_terms(
    df,
    semester_col="Semester",
    store_col="StoreCode",
    course_col="Course_Name",
    academic_year=None,
):
    out = df.copy()
    #normalize text
    sem = out[semester_col].astype(str).str.strip()
    store = out[store_col].astype(str).str.strip().str.upper()

    #academic year
    year_pat = re.compile(r"^\s*\d{4}\s*-\s*\d{4}\s*$")
    out["AcademicYear"] = np.where(
        sem.str.match(year_pat),
        sem,
        academic_year
    )
    #elective flag in Semester 1&2 for elective subject in the Sem
    out["IsElective"] = sem.isin(["Semester 1", "Semester 2"]).astype(int)

    #exapnding to store in AssessmentinTerm with termnumber
    out["AssessmentinTerm"] = store.str[0].where(store.str[0].isin(["S","E","F"]))
    out["TermNumber"] = store.str.extract(r"(\d)$")[0].astype("float")
    out["TermNumber"] = out["TermNumber"].where(out["TermNumber"].isin([1.0, 2.0]))

    #Label Term for all rows for Seemester
    out["Term"] = np.where(out["TermNumber"].eq(1), "Semester 1",
                    np.where(out["TermNumber"].eq(2), "Semester 2", None))
    return out

df_new_25 = normalize_terms(df_new_25, academic_year="2024-2025")

#outliers
numeric_cols = df_new_25.select_dtypes(include=['number']).columns
#remove outliers
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    lower= Q1 - 1.5 * (Q3 - Q1)
    upper = Q3 + 1.5 * (Q3 - Q1)
    return df[(df[column] >= lower) & (df[column] <= upper)]
    for col in numeric_cols:
        df_new_25 = remove_outliers_iqr(df_new_25, col)
df_new_25= df_new_25.dropna(subset=['Grade']).copy()

#labels from cureent percent for risk catgories
#Severe <65, Low 65-75, Moderate 75-85, High 85-95, Excelling ≥95

edges  = [-np.inf, 65, 75, 85, 95, np.inf]
labels = ["Severe", "Low", "Moderate", "High", "Excelling"]

df_new_25["Percent"] = pd.to_numeric(df_new_25["Percent"], errors="coerce").clip(0, 100)
df_new_25["risk_score"] = pd.cut(
    df_new_25["Percent"], bins=edges, labels=labels,
    right=False, include_lowest=True, ordered=True
).astype(pd.CategoricalDtype(categories=labels, ordered=True))
risk_order = {"Severe":0, "Low":1, "Moderate":2, "High":3, "Excelling":4}
df_new_25['risk_ordinal'] = df_new_25['risk_score'].map(risk_order)
df_new_25['Grade_Level'] = df_new_25['Grade_Level'].replace({9: 'Grade 9', 10: 'Grade 10', 11: 'Grade 11', 12: 'Grade 12'})

In [4]:
df_new_25.to_csv('24-25_6.csv', index=False)

In [ ]:
data_new.head()

,Student_Number,Gender,Ethnicity,Grade_Level,Course_Name,Grade,Percent,AcademicYear,Semester,StoreCode,EarnedCrHrs,IsElective,AssessmentinTerm,TermNumber,Term,risk_score
0,2534,F,Bah,Grade 12,English Language & Literature,B-,81,2024-2025,2024-2025,F1,0.5,0,F,1.0,Semester 1,Moderate
1,2510,M,Sau,Grade 12,IB Business Management SL,A,96,2024-2025,2024-2025,F1,0.5,0,F,1.0,Semester 1,Excelling
2,2576,M,Bah,Grade 12,English Language & Literature,B,86,2024-2025,2024-2025,F1,0.5,0,F,1.0,Semester 1,High
3,2405,M,Bah,Grade 12,Integrated Math,C,73,2024-2025,2024-2025,F1,0.5,0,F,1.0,Semester 1,Low
4,2586,F,Bah,Grade 12,IB English A Language & Literature SL,A,93,2024-2025,2024-2025,F1,0.5,0,F,1.0,Semester 1,High
